In [11]:
# ============================================================
# CELL 1 — Installs, imports, prompt, config
# ============================================================

!pip install -U google-genai openpyxl pandas -q

from google import genai
from google.genai import types
from google.colab import userdata
import pandas as pd
import json, time

client     = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
GEMINI_MODEL = "gemini-2.5-flash-lite"

SYSTEM_PROMPT = (
    "You are a fact-checking analyst. "
    "Your role is to write a concise analytical note about a political claim — "
    "surfacing what it specifically says, what context is needed to understand it, "
    "and where it may be ambiguous — without judging whether it is true or false."
)

USER_TEMPLATE = """Write a short analytical note about the political claim below.

Guidelines:
- Engage directly with the specific content of the claim: the named person, the figure cited, the policy described, the comparison drawn. Make that content the subject of your note.
- If the claim attributes a statement or action to a named person, open by naming that person and what they said or did. Do not open with a generic description of the claim.
- Where a term or figure in the claim could be read in more than one way, say what those two readings are concretely — do not say that interpretation 'depends on' something without specifying what the two outcomes would be.
- You may use contrast ('but', 'however') to show two sides of the same element.
- Do not evaluate accuracy. Banned evaluative phrasings: 'correctly states', 'misleadingly claims', 'is accurate', 'is inaccurate', 'is true', 'is false'.
- Do not use procedural language: write the relevant considerations directly, do not describe what a fact-checker 'would need to' or 'should' verify.
- Maintain a neutral, analytical tone.

Strictly forbidden — never use any of the following words or phrases:
- "ambiguity", "ambiguous", "ambiguously"
- "key ambiguity", "central ambiguity", "core ambiguity"
- "hinges on", "turns on", "centers on"
- "The assertion", "The claim asserts", "The claim presents", "The claim suggests", "The claim states", "The claim implies"
- "The statement", "This statement"
- "Relevant context"
- "Additionally", "Furthermore", "Moreover"
- "Implicit assumption"
- "Understanding this claim requires", "Requires clarity on"
- "One must consider", "It is worth considering", "It is important to note"
- "depends on how", "depends on what", "depends on whether", "depends on the"
- "would need to be defined", "would need to be understood" "rather than"

Do not follow a fixed structure. Every note must differ from the others in how it is organised. Some notes open with the speaker's name; others with the specific figure or date. Some address a single pivot in two or three sentences; others trace a narrower point in one sentence. Never apply the same sentence-by-sentence template twice. Vary how each note is organised — in the opening, in the number of sentences, and in how you close. Do not always close by generalising about interpretation.

Length: Between 30 and 90 words. Match length strictly to complexity — a single-figure claim warrants 30–45 words; a claim with multiple interacting conditions warrants 70–90 words. Do not pad.

Output: A single paragraph. No bullet points, headers, or numbered lists.

Claim: {claim}

Analytical note:"""

print("✅ Cell 1 OK")

✅ Cell 1 OK


In [12]:
# ============================================================
# CELL 2 — Load data + build JSONL + upload
# ============================================================

df = pd.read_excel("claims_for_api.xlsx")
df.columns = [c.strip() for c in df.columns]

# df = df.iloc[:2100]          # Part 1      CHANGEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEE
# df = df.iloc[2100:4200]    # Part 2
# df = df.iloc[4200:6300]    # Part 3
# df = df.iloc[6300:8400]    # Part 4
# df = df.iloc[8400:10500]   # Part 5
df = df.iloc[10500:]       # Part 6

claim_col = "statement"
print(f"Rows loaded: {len(df)}")

# Build JSONL
jsonl_path = "gemini_batch_input.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for idx, row in df.iterrows():
        claim = str(row[claim_col]).strip()
        task = {
            "key": f"row_{idx}",
            "request": {
                "systemInstruction": {        # camelCase obbligatorio
                    "parts": [{"text": SYSTEM_PROMPT}]
                },
                "contents": [{
                    "role": "user",
                    "parts": [{"text": USER_TEMPLATE.format(claim=claim)}]
                }],
                "generationConfig": {         # camelCase obbligatorio
                    "temperature": 0.7,
                    "maxOutputTokens": 2048
                }
            }
        }
        f.write(json.dumps(task, ensure_ascii=False) + "\n")

print(f"✅ JSONL scritto: {jsonl_path}")

# Upload — mime_type='text/plain' (application/jsonl ha un bug noto nell'SDK)
print("Uploading file...")
uploaded_file = client.files.upload(
    file=jsonl_path,
    config=types.UploadFileConfig(
        display_name="gemini-batch-justifications",
        mime_type="text/plain"
    )
)
print(f"✅ Upload OK")
print(f"   Name: {uploaded_file.name}")
print(f"   URI:  {uploaded_file.uri}")

Rows loaded: 2291
✅ JSONL scritto: gemini_batch_input.jsonl
Uploading file...
✅ Upload OK
   Name: files/twed9ijjdrzy
   URI:  https://generativelanguage.googleapis.com/v1beta/files/twed9ijjdrzy


In [13]:
# ============================================================
# CELL 3 — Submit batch job
# ============================================================

print("Submitting batch job...")
batch_job = client.batches.create(
    model=GEMINI_MODEL,
    src=uploaded_file.name,       # .name non .uri ← era il bug principale
    config={"display_name": "justification-batch"}
)

GEMINI_BATCH_NAME = batch_job.name
print(f"🚀 Job creato!")
print(f"   Name:  {GEMINI_BATCH_NAME}")
print(f"   State: {batch_job.state}")

Submitting batch job...
🚀 Job creato!
   Name:  batches/s8zr6tpt6pl0nt3t9nrvm8hajg4s084j87q5
   State: JobState.JOB_STATE_PENDING


In [16]:
# ============================================================
# CELL 4 — Polling fino a completamento
# ============================================================
from google import genai
from google.genai import types
from google.colab import userdata
import pandas as pd
import json, time

client     = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))


# Se Colab si disconnette, incolla qui il batch name:
GEMINI_BATCH_NAME = "batches/s8zr6tpt6pl0nt3t9nrvm8hajg4s084j87q5"        # changeeeeeeeeeeee !!!!!!!!!!!!!!!

TERMINAL_STATES = {"JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED", "JOB_STATE_CANCELLED"}

print(f"Polling: {GEMINI_BATCH_NAME}\n")
while True:
    job   = client.batches.get(name=GEMINI_BATCH_NAME)
    state = job.state.name
    print(f"  [{time.strftime('%H:%M:%S')}] {state}")

    if state in TERMINAL_STATES:
        break
    time.sleep(30)

if state == "JOB_STATE_SUCCEEDED":
    print("\n✅ Completato!")
else:
    raise RuntimeError(f"Job terminato con stato: {state}\nErrore: {getattr(job, 'error', 'N/A')}")

Polling: batches/s8zr6tpt6pl0nt3t9nrvm8hajg4s084j87q5

  [15:50:11] JOB_STATE_SUCCEEDED

✅ Completato!


In [17]:
# ============================================================
# CELL 5 — Download + parse + salva Excel
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter


df = pd.read_excel("claims_for_api.xlsx")
df.columns = [c.strip() for c in df.columns]
claim_col = "statement"
print(f"Rows loaded: {len(df)}")

# df = df.iloc[:2100]          # Part 1                        # CHANGEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEE
# df = df.iloc[2100:4200]    # Part 2
# df = df.iloc[4200:6300]    # Part 3
# df = df.iloc[6300:8400]    # Part 4
# df = df.iloc[8400:10500]   # Part 5
df = df.iloc[10500:]       # Part 6


# Download risultati
result_bytes = client.files.download(file=job.dest.file_name)
result_text  = result_bytes.decode("utf-8")

# Parse
results = []
for line in result_text.splitlines():
    if not line.strip():
        continue
    obj = json.loads(line)
    key = obj.get("key", "")
    try:
        text   = obj["response"]["candidates"][0]["content"]["parts"][0]["text"].strip()
        status = "success"
    except (KeyError, IndexError) as e:
        text   = str(obj.get("error", e))
        status = "error"
    results.append({"key": key, "status": status, "Gemini_Justification": text})

results_df = pd.DataFrame(results)
print(f"Successi: {(results_df['status']=='success').sum()}/{len(results_df)}")
print(f"Errori:   {(results_df['status']=='error').sum()}/{len(results_df)}")

# Merge con df originale
df_out = df.copy()
df_out["key"] = [f"row_{i}" for i in df.index]
df_out = df_out.merge(results_df[["key", "Gemini_Justification"]], on="key", how="left")
df_out = df_out.drop(columns=["key"])
df_out = df_out.rename(columns={claim_col: "Claim"})


# Salva Excel
output_path = "gemini_just_on_original_part6.xlsx"                         # CHANGEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEE
df_out[["Claim", "Gemini_Justification"]].to_excel(output_path, index=False, sheet_name="Results")

wb = load_workbook(output_path)
ws = wb["Results"]

header_fill = PatternFill("solid", fgColor="1A5276")
header_font = Font(bold=True, color="FFFFFF", name="Arial", size=11)
cell_font   = Font(name="Arial", size=10)
wrap_align  = Alignment(wrap_text=True, vertical="top")

col_widths = [70, 80]
for col_idx, (cell, width) in enumerate(zip(ws[1], col_widths), start=1):
    cell.fill      = header_fill
    cell.font      = header_font
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.column_dimensions[get_column_letter(col_idx)].width = width

ws.row_dimensions[1].height = 30
for row in ws.iter_rows(min_row=2):
    for cell in row:
        cell.font      = cell_font
        cell.alignment = wrap_align

ws.freeze_panes = "A2"
wb.save(output_path)
print(f"\n✅ Salvato: {output_path}")

for i, row in df_out.head(3).iterrows():
    print(f"\n─── Row {i} ───")
    print(f"CLAIM:  {str(row['Claim'])[:100]}")
    print(f"GEMINI: {str(row['Gemini_Justification'])[:200]}")

Rows loaded: 12791
Successi: 2291/2291
Errori:   0/2291

✅ Salvato: gemini_just_on_original_part6.xlsx

─── Row 0 ───
CLAIM:  On Medicare for current retirees, hes cutting $716 billion from the program.
GEMINI: This statement attributes a reduction of $716 billion to Medicare for current retirees, allegedly due to actions by a specific individual. The figure could refer to proposed budget cuts or actual spen

─── Row 1 ───
CLAIM:  Sen. Marco Rubio refuses to accept the basic science on climate change and is a climate change denie
GEMINI: Senator Marco Rubio is described as refusing to accept basic climate science and being a climate change denier. This characterization can be understood in two ways: either Rubio disputes the scientifi

─── Row 2 ───
CLAIM:  There are 10 or 20 deaths a year from foodborne illness in the United States.
GEMINI: The claim states that there are 10 to 20 deaths annually from foodborne illness in the United States. This figure could refer to confirmed deaths 